# 🛠️ Network Build Checker (v2): works with or without real devices

**What it does, in plain English:** you describe (or let it learn) how your network *should* look. It then logs in to every router/switch, asks them what they are actually doing, compares the two, and tells you **what is wrong, why, and what command to try next**.

Think of it as the final inspector at the end of a factory line: the drawing is the *design*, the device is the *product*, and this notebook is the inspector with a checklist and a red pen.

### Start here (3 steps)
1. Run every cell once (**Run → Run All**). The dashboard appears at the bottom of section 6.
2. Stay on **🧪 Practice lab**. It is a *virtual network that really computes OSPF, routes and pings*, so nothing needs to be installed or connected.
3. Pick a fault from the list → **💥 Break it** → **✅ Run checks**. Read the diagnosis, then **🔧 Repair all** and run again.

When you are ready for the real thing, switch to **🔌 My real devices**, fill in the inventory, and press the same buttons.

| What is checked | Question asked | Command sent |
|---|---|---|
| 🟦 VLANs | Do the planned VLANs exist with the right names? | `show vlan brief` |
| 🟩 OSPF neighbours | Is every neighbour in `FULL` state? | `show ip ospf neighbor` |
| 🟨 Routes | Were the expected routes learned via OSPF? | `show ip route` |
| 🟪 Ping | Can device A really reach address B? | `ping` |

## 1. Install and import

In [1]:
%pip install -q ipywidgets netmiko

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, json, time, heapq, html, ipaddress
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML, clear_output

WORKDIR = Path("netcheck_data"); WORKDIR.mkdir(exist_ok=True)
REPORT_DIR = WORKDIR / "reports"; REPORT_DIR.mkdir(exist_ok=True)
print("Ready. Files will be saved in:", WORKDIR.resolve())

Ready. Files will be saved in: C:\Users\praha\Downloads\netcheck_data


## 2. Your network, described once

`INVENTORY` lists the devices, `LINKS` how they connect (only used for the picture), and `PING_TARGETS` the important addresses that must stay reachable.
For real devices you can edit this as JSON right inside the dashboard, no code changes needed.

```
 [SW1] ── [R1] ──10.0.12.0/30── [R2] ──10.0.23.0/30── [R3]
 VLAN 10/20/30  LAN 192.168.10.0/24              LAN 192.168.30.0/24
```

In [3]:
INVENTORY = {
    "SW1": {"role": "switch", "host": "10.0.0.11", "port": 22, "device_type": "cisco_ios", "pos": [70, 120]},
    "R1":  {"role": "router", "router_id": "1.1.1.1", "host": "10.0.0.1", "port": 22, "device_type": "cisco_ios", "pos": [230, 120]},
    "R2":  {"role": "router", "router_id": "2.2.2.2", "host": "10.0.0.2", "port": 22, "device_type": "cisco_ios", "pos": [390, 120]},
    "R3":  {"role": "router", "router_id": "3.3.3.3", "host": "10.0.0.3", "port": 22, "device_type": "cisco_ios", "pos": [550, 120]},
}
LINKS = [("SW1", "R1"), ("R1", "R2"), ("R2", "R3")]
PING_TARGETS = ["10.0.12.1", "10.0.12.2", "10.0.23.1", "10.0.23.2", "192.168.10.1", "192.168.30.1"]

## 3. Reading device output (parsers)

Devices answer in plain text. These small functions pull out just the facts we need. Self-tests at the bottom make sure they work before we trust them.

In [4]:
def parse_vlans(text):
    """'show vlan brief' -> {vlan_id: name}"""
    out = {}
    for line in text.splitlines():
        m = re.match(r"^(\d+)\s+(\S+)\s+(active|act/unsup|suspended)", line)
        if m:
            out[int(m.group(1))] = m.group(2)
    return out

def parse_ospf_neighbors(text):
    """'show ip ospf neighbor' -> {router_id: state}"""
    out = {}
    for line in text.splitlines():
        m = re.match(r"^(\d+\.\d+\.\d+\.\d+)\s+\d+\s+([A-Z0-9]+)(?:/\S+)?\s+\S+\s+\d+\.\d+\.\d+\.\d+\s+\S+", line)
        if m:
            out[m.group(1)] = m.group(2)
    return out

def parse_routes(text):
    """'show ip route' -> {prefix: code}  (C connected, L local, O OSPF ...)"""
    out = {}
    for line in text.splitlines():
        m = re.match(r"^([A-Z\*]{1,2})(?:\s+[A-Z]{1,2}\d?)?\s+(\d+\.\d+\.\d+\.\d+/\d+)", line)
        if m:
            out[m.group(2)] = m.group(1).replace("*", "")
    return out

def parse_ping(text):
    """ping output -> success percent"""
    m = re.search(r"Success rate is (\d+) percent", text)
    return int(m.group(1)) if m else 0

## 4. The practice lab: a tiny virtual network that *really works*

This is not pre-typed fake output. `VirtualNetwork` keeps the state of every cable, interface, VLAN and router, then **calculates** what each command would show:
- OSPF neighbour states come from the link conditions (cable, MTU, timers).
- Routes come from a shortest-path calculation over the adjacencies that are actually `FULL`, which is how OSPF itself thinks.
- A ping succeeds only if the forward path **and** the return path exist, hop by hop.

So if you cut one cable, many things change together, exactly like a real network.

In [5]:
class VirtualNetwork:
    PORTS = {10: "Fa0/3, Fa0/4", 20: "Fa0/5, Fa0/6", 30: "Fa0/7"}

    def __init__(self):
        self.reset()

    def reset(self):
        self.router_up = {"R1": True, "R2": True, "R3": True}
        self.switch_up = True
        self.rid = {"R1": "1.1.1.1", "R2": "2.2.2.2", "R3": "3.3.3.3"}
        self.links = {
            "R1-R2": dict(a="R1", b="R2", net="10.0.12.0/30", up=True,
                          ip={"R1": "10.0.12.1", "R2": "10.0.12.2"},
                          intf={"R1": "GigabitEthernet0/1", "R2": "GigabitEthernet0/0"},
                          mtu={"R1": 1500, "R2": 1500}, hello={"R1": 10, "R2": 10}),
            "R2-R3": dict(a="R2", b="R3", net="10.0.23.0/30", up=True,
                          ip={"R2": "10.0.23.1", "R3": "10.0.23.2"},
                          intf={"R2": "GigabitEthernet0/1", "R3": "GigabitEthernet0/0"},
                          mtu={"R2": 1500, "R3": 1500}, hello={"R2": 10, "R3": 10}),
        }
        self.lans = {
            "R1": dict(net="192.168.10.0/24", ip="192.168.10.1", intf="GigabitEthernet0/0", up=True),
            "R3": dict(net="192.168.30.0/24", ip="192.168.30.1", intf="GigabitEthernet0/1", up=True),
        }
        self.vlans = {10: "SALES", 20: "HR", 30: "SERVERS"}

    # ---- how OSPF sees each link --------------------------------------
    def adj_state(self, L):
        if not (L["up"] and self.router_up[L["a"]] and self.router_up[L["b"]]):
            return None                      # no cable / device off: no neighbour at all
        if L["hello"][L["a"]] != L["hello"][L["b"]]:
            return None                      # hello timers differ: hellos are ignored
        if L["mtu"][L["a"]] != L["mtu"][L["b"]]:
            return "EXSTART"                 # MTU mismatch: stuck negotiating
        return "FULL"

    def connected(self, r):
        """(network, my_ip, interface, peer_router) for everything r is directly attached to"""
        items = []
        for L in self.links.values():
            if r in (L["a"], L["b"]) and L["up"]:
                peer = L["b"] if r == L["a"] else L["a"]
                items.append((L["net"], L["ip"][r], L["intf"][r], peer))
        if r in self.lans and self.lans[r]["up"]:
            l = self.lans[r]; items.append((l["net"], l["ip"], l["intf"], None))
        return items

    def routing_table(self, r):
        table = {}
        for net, ip, intf, peer in self.connected(r):
            table[net] = dict(code="C", intf=intf, via=None, metric=0, nh=None, peer=peer, ip=ip)
        dist, first, pq = {r: 0}, {r: None}, [(0, r)]
        while pq:                                           # shortest path over FULL adjacencies
            d, u = heapq.heappop(pq)
            if d > dist[u]:
                continue
            for L in self.links.values():
                if u not in (L["a"], L["b"]) or self.adj_state(L) != "FULL":
                    continue
                v = L["b"] if u == L["a"] else L["a"]
                if d + 1 < dist.get(v, 99):
                    dist[v] = d + 1
                    first[v] = first[u] if u != r else (v, L["ip"][v], L["intf"][r], L["ip"][r])
                    heapq.heappush(pq, (d + 1, v))
        for x, dx in dist.items():
            if x == r:
                continue
            for net, _, _, _ in self.connected(x):
                if net in table and table[net]["code"] == "C":
                    continue
                m = dx + 1
                if net not in table or m < table[net]["metric"]:
                    nh, via, intf, myip = first[x]
                    table[net] = dict(code="O", intf=intf, via=via, metric=m, nh=nh, peer=None, ip=myip)
        return table

    # ---- forwarding ------------------------------------------------------
    def owner_of(self, ip):
        for L in self.links.values():
            for r, a in L["ip"].items():
                if a == ip: return r
        for r, l in self.lans.items():
            if l["ip"] == ip: return r
        return None

    def lookup(self, r, dst):
        if any(ip == dst for _, ip, _, _ in self.connected(r)):
            return "local"
        best = None
        for prefix, e in self.routing_table(r).items():
            net = ipaddress.ip_network(prefix)
            if ipaddress.ip_address(dst) in net and (best is None or net.prefixlen > best[0].prefixlen):
                best = (net, e)
        return best[1] if best else None

    def trace(self, src, dst):
        cur, first_ip, hops = src, None, 0
        for _ in range(10):
            if not self.router_up[cur]:
                return False, first_ip, hops
            e = self.lookup(cur, dst)
            if e == "local":
                return True, first_ip, hops
            if e is None:
                return False, first_ip, hops
            first_ip = first_ip or e["ip"]
            hops += 1
            cur = e["peer"] if e["code"] == "C" else e["nh"]
            if cur is None:
                return False, first_ip, hops
        return False, first_ip, hops

    def ping_result(self, src, dst):
        ok, src_ip, hops = self.trace(src, dst)
        if not ok:
            return 0, 0
        owner = self.owner_of(dst)
        if src_ip and owner and owner != src:
            back, _, _ = self.trace(owner, src_ip)       # the reply must find its way home too
            if not back:
                return 0, 0
        return 100, max(1, hops)

    # ---- the "command line" -------------------------------------------
    def cli(self, dev, command):
        command = command.strip()
        if dev == "SW1":
            if not self.switch_up:
                raise ConnectionError("SW1: connection timed out")
            return self._vlan_text() if command.startswith("show vlan brief") else "% Invalid input detected at '^' marker."
        if not self.router_up.get(dev, False):
            raise ConnectionError(f"{dev}: connection timed out")
        if command.startswith("show ip ospf neighbor"):
            return self._ospf_text(dev)
        if command.startswith("show ip route"):
            return self._route_text(dev)
        if command.startswith("ping"):
            return self._ping_text(dev, command.split()[1])
        return "% Invalid input detected at '^' marker."

    def _vlan_text(self):
        rows = ["VLAN Name                             Status    Ports",
                "---- -------------------------------- --------- -------------------------------",
                f"{1:<4} {'default':<32} {'active':<9} Gi0/1, Gi0/2"]
        for vid, name in sorted(self.vlans.items()):
            rows.append(f"{vid:<4} {name:<32} {'active':<9} {self.PORTS.get(vid, '')}")
        rows.append(f"{1002:<4} {'fddi-default':<32} {'act/unsup':<9}")
        return "\n".join(rows)

    def _ospf_text(self, r):
        rows = ["Neighbor ID     Pri   State           Dead Time   Address         Interface"]
        for L in self.links.values():
            if r not in (L["a"], L["b"]):
                continue
            st = self.adj_state(L)
            if st is None:
                continue
            peer = L["b"] if r == L["a"] else L["a"]
            role = ("DR" if r == L["b"] else "BDR") if st == "FULL" else "-"
            rows.append(f"{self.rid[peer]:<15} {1:>3}   {st + '/' + role:<15} {'00:00:34':<11} {L['ip'][peer]:<15} {L['intf'][r]}")
        return "\n".join(rows)

    def _route_text(self, r):
        rows = ["Codes: L - local, C - connected, S - static, O - OSPF", "Gateway of last resort is not set", ""]
        for net, ip, intf, _ in self.connected(r):
            rows.append(f"C        {net} is directly connected, {intf}")
            rows.append(f"L        {ip}/32 is directly connected, {intf}")
        for prefix, e in self.routing_table(r).items():
            if e["code"] == "O":
                rows.append(f"O        {prefix} [110/{e['metric']}] via {e['via']}, 00:{4 + e['metric']:02d}:12, {e['intf']}")
        return "\n".join(rows)

    def _ping_text(self, r, dst):
        pct, hops = self.ping_result(r, dst)
        head = f"Type escape sequence to abort.\nSending 5, 100-byte ICMP Echos to {dst}, timeout is 2 seconds:\n"
        if pct == 0:
            return head + ".....\nSuccess rate is 0 percent (0/5)"
        return head + f"!!!!!\nSuccess rate is 100 percent (5/5), round-trip min/avg/max = 1/{hops}/{hops + 2} ms"


# ----- things you can break (each one is a classic real-world fault) -----
def _cable(n):  n.links["R2-R3"]["up"] = False
def _mtu(n):    n.links["R2-R3"]["mtu"]["R3"] = 1400
def _hello(n):  n.links["R1-R2"]["hello"]["R2"] = 30
def _r3off(n):  n.router_up["R3"] = False
def _lan(n):    n.lans["R3"]["up"] = False
def _vmiss(n):  n.vlans.pop(30, None)
def _vname(n):  n.vlans[20] = "HR-DEPT"
def _swoff(n):  n.switch_up = False

FAULTS = {
    "cable":  ("🔌 Cable cut between R2 and R3", _cable),
    "mtu":    ("📏 MTU mismatch on the R2–R3 link", _mtu),
    "hello":  ("⏱️ OSPF hello-timer mismatch on R1–R2", _hello),
    "r3off":  ("⚡ R3 powers off", _r3off),
    "lan":    ("🚫 R3's LAN interface shut down", _lan),
    "vmiss":  ("🗑️ VLAN 30 deleted on SW1", _vmiss),
    "vname":  ("✏️ VLAN 20 renamed on SW1", _vname),
    "swoff":  ("⚡ SW1 unreachable", _swoff),
}

## 5. Connecting to devices

Both the virtual lab and real hardware are wrapped to look identical to the checker: `with backend.open("R1") as s: s.run("show ...")`.
Real devices use **Netmiko**. Passwords are typed into a hidden box or read from environment variables (`NET_USER`, `NET_PASS`, `NET_ENABLE`), never stored in the notebook.

In [6]:
class _SimSession:
    def __init__(self, net, name): self.net, self.name = net, name
    def __enter__(self): return self
    def __exit__(self, *a): return False
    def run(self, cmd): return self.net.cli(self.name, cmd)

class SimBackend:
    label = "Practice lab"
    def __init__(self, net): self.net = net
    def open(self, name): return _SimSession(self.net, name)

class _LiveSession:
    def __init__(self, backend, name): self.b, self.name, self.conn = backend, name, None
    def __enter__(self):
        from netmiko import ConnectHandler
        spec = self.b.inventory[self.name]
        self.conn = ConnectHandler(device_type=spec.get("device_type", "cisco_ios"), host=spec["host"],
                                   port=spec.get("port", 22), username=self.b.user, password=self.b.password,
                                   secret=self.b.secret, timeout=20)
        if self.b.secret:
            self.conn.enable()
        return self
    def run(self, cmd): return self.conn.send_command(cmd, read_timeout=30)
    def __exit__(self, *a):
        if self.conn: self.conn.disconnect()
        return False

class LiveBackend:
    label = "Live devices"
    def __init__(self, inventory, user, password, secret=""):
        self.inventory, self.user, self.password, self.secret = inventory, user, password, secret
    def open(self, name): return _LiveSession(self, name)

## 6. Learn the design, run the checks, explain the problems

**Learn design** is the user-friendly part: instead of typing every expected VLAN, neighbour and route, point the tool at a *healthy* network and it records what "good" looks like into `design_*.json`. Later runs are compared against that file. (You can still edit the file by hand.)

In [7]:
def learn_design(backend, inventory):
    design = {}
    for name, spec in inventory.items():
        with backend.open(name) as s:
            if spec["role"] == "switch":
                v = parse_vlans(s.run("show vlan brief"))
                design[name] = {"vlans": {str(i): n for i, n in v.items() if 2 <= i <= 1001}}
            else:
                nb = parse_ospf_neighbors(s.run("show ip ospf neighbor"))
                rt = parse_routes(s.run("show ip route"))
                pings = []
                for t in PING_TARGETS:
                    if rt.get(f"{t}/32") == "L":
                        continue                                   # skip the router's own addresses
                    if parse_ping(s.run(f"ping {t} repeat 5")) >= 80:
                        pings.append({"target": t, "min_success": 80})
                design[name] = {"ospf_neighbors": [r for r, st in nb.items() if st == "FULL"],
                                "routes": {p: c for p, c in rt.items() if c == "O"},
                                "pings": pings}
    return design


@dataclass
class Result:
    device: str; category: str; check: str; expected: str; actual: str; status: str


def check_vlans(s, name, exp):
    actual, out = parse_vlans(s.run("show vlan brief")), []
    for vid, want in exp.items():
        vid = int(vid); got = actual.get(vid)
        if got is None:
            out.append(Result(name, "VLAN", f"VLAN {vid}", want, "missing", "FAIL"))
        elif got.upper() != want.upper():
            out.append(Result(name, "VLAN", f"VLAN {vid} name", want, got, "FAIL"))
        else:
            out.append(Result(name, "VLAN", f"VLAN {vid}", want, got, "PASS"))
    return out

def check_ospf(s, name, exp):
    actual = parse_ospf_neighbors(s.run("show ip ospf neighbor"))
    return [Result(name, "OSPF", f"Neighbor {rid}", "FULL", actual.get(rid, "not seen"),
                   "PASS" if actual.get(rid) == "FULL" else "FAIL") for rid in exp]

def check_routes(s, name, exp):
    actual, out = parse_routes(s.run("show ip route")), []
    for prefix, code in exp.items():
        got = actual.get(prefix)
        out.append(Result(name, "Route", prefix, f"via {code}", f"via {got}" if got else "absent",
                          "PASS" if got == code else "FAIL"))
    return out

def check_pings(s, name, exp):
    out = []
    for p in exp:
        pct = parse_ping(s.run(f"ping {p['target']} repeat 5"))
        out.append(Result(name, "Ping", f"to {p['target']}", f">= {p['min_success']}%", f"{pct}%",
                          "PASS" if pct >= p["min_success"] else "FAIL"))
    return out

def error_rows(name, exp, why):
    rows = [Result(name, "VLAN", f"VLAN {v}", n, "device unreachable", "ERROR") for v, n in exp.get("vlans", {}).items()]
    rows += [Result(name, "OSPF", f"Neighbor {r}", "FULL", "device unreachable", "ERROR") for r in exp.get("ospf_neighbors", [])]
    rows += [Result(name, "Route", p, f"via {c}", "device unreachable", "ERROR") for p, c in exp.get("routes", {}).items()]
    rows += [Result(name, "Ping", f"to {p['target']}", f">= {p['min_success']}%", "device unreachable", "ERROR") for p in exp.get("pings", [])]
    return rows or [Result(name, "Connection", "login", "reachable", why[:60], "ERROR")]

def run_checks(backend, design):
    rows, t0 = [], time.time()
    for name, exp in design.items():
        try:
            tmp = []
            with backend.open(name) as s:                     # one login per device
                if "vlans" in exp:           tmp += check_vlans(s, name, exp["vlans"])
                if "ospf_neighbors" in exp:  tmp += check_ospf(s, name, exp["ospf_neighbors"])
                if "routes" in exp:          tmp += check_routes(s, name, exp["routes"])
                if "pings" in exp:           tmp += check_pings(s, name, exp["pings"])
            rows += tmp
        except Exception as e:                                # one dead device never stops the run
            rows += error_rows(name, exp, str(e))
    return pd.DataFrame([asdict(r) for r in rows]), time.time() - t0

### The "what went wrong?" engine

A pile of red rows is hard to read. This step groups them into **root causes** (fix these) and **side-effects** (they clear up by themselves), then suggests the next commands to type. It is a small rule-based expert system today; the structure (symptoms → cause) is also ready to be replaced by a trained classifier later.

In [8]:
def _dead_devices(df):
    return sorted(df[df.status == "ERROR"].device.unique()) if not df.empty else []

def bad_pairs(df, inventory):
    """Router pairs whose OSPF adjacency is failing (excluding pairs explained by a dead device)."""
    rid2dev = {v["router_id"]: k for k, v in inventory.items() if v.get("router_id")}
    dead, pairs = _dead_devices(df), {}
    if df.empty: return pairs
    for r in df[(df.category == "OSPF") & (df.status == "FAIL")].itertuples():
        peer = rid2dev.get(r.check.replace("Neighbor ", ""), r.check)
        if peer in dead: continue
        pairs.setdefault(frozenset([r.device, peer]), (r.device, peer, r.actual))
    return pairs

def diagnose(df, inventory):
    items = []
    if df.empty: return items
    dead = _dead_devices(df)
    for d in dead:
        items.append(dict(kind="root", title=f"{d} cannot be reached",
            why="The checker could not log in. The device may be powered off, have a wrong IP/port, SSH disabled, or wrong credentials.",
            fix=[f"ping {inventory.get(d, {}).get('host', d)} from your PC", "check power / console / SSH settings and the username & password"]))
    pairs = bad_pairs(df, inventory)
    for a, b, state in pairs.values():
        if state in ("EXSTART", "EXCHANGE", "LOADING"):
            items.append(dict(kind="root", title=f"OSPF between {a} and {b} is stuck negotiating ({state})",
                why="The two routers see each other but cannot finish exchanging their databases. This is the classic sign of an MTU mismatch on that link.",
                fix=["show interface <intf> | include MTU   (run on both routers and compare)", "make both sides equal (ip mtu 1500) or add: ip ospf mtu-ignore"]))
        elif state in ("INIT", "2WAY"):
            items.append(dict(kind="root", title=f"{a} hears {b} but {b} does not hear {a} (state {state})",
                why="Hellos flow in only one direction. Usual suspects: an ACL or firewall dropping OSPF (protocol 89), or a one-way link fault.",
                fix=["show ip access-lists", "show ip ospf interface   (look at Hello/Dead on both ends)"]))
        else:
            items.append(dict(kind="root", title=f"{a} and {b} are not forming an OSPF neighbourship at all",
                why="No adjacency was seen. Likely a cable/interface down, different hello/dead timers, different area or network type, or a passive interface.",
                fix=["show ip interface brief   (is the interface up/up?)", "show ip ospf interface   (compare Hello/Dead timers, area and network type on both ends)"]))
    for r in df[(df.category == "VLAN") & (df.status == "FAIL")].itertuples():
        vid = re.search(r"VLAN (\d+)", r.check).group(1)
        what = f"VLAN {vid} is missing" if r.actual == "missing" else f"VLAN {vid} is named '{r.actual}' but the design says '{r.expected}'"
        items.append(dict(kind="root", title=f"{r.device}: {what}",
            why="The switch configuration does not match the design.", fix=[f"vlan {vid}", f" name {r.expected}"]))
    fails = df[df.category.isin(["Route", "Ping"]) & (df.status == "FAIL")]
    if len(fails):
        if dead or pairs:
            items.append(dict(kind="symptom", title=f"{len(fails)} route/ping failures look like side-effects",
                why="Routes and pings depend on OSPF working. They should recover on their own once the problems above are fixed.",
                fix=["fix the red items above, then press ✅ Run checks again"]))
        else:
            for dev, g in fails.groupby("device"):
                items.append(dict(kind="root", title=f"{dev}: {len(g)} route/ping checks fail but OSPF looks healthy",
                    why="Routing neighbours are fine, so look at what is *advertised* or *blocked*: a LAN interface down, a missing network statement, a passive interface or an ACL.",
                    fix=["show ip interface brief", "show run | section ospf", "show ip access-lists"]))
    items.sort(key=lambda x: 0 if x["kind"] == "root" else 1)
    return items

### Making the results easy on the eyes
Scorecard, colour-coded table, diagnosis cards and a live topology picture (green = healthy, red = broken, amber = unreachable). The same pieces are reused for the saved HTML report.

In [9]:
STY = {"PASS": ("#d4edda", "#155724", "✅"), "FAIL": ("#f8d7da", "#721c24", "❌"), "ERROR": ("#fff3cd", "#856404", "⚠️")}
FONT = "font-family:Segoe UI,Arial,sans-serif;"

def h_scorecard(df, elapsed, label):
    n = len(df)
    if n == 0: return "<p>No checks yet.</p>"
    p, f, e = [(df.status == s).sum() for s in ("PASS", "FAIL", "ERROR")]
    pct = round(100 * p / n)
    color = "#28a745" if pct == 100 else ("#e0a800" if pct >= 70 else "#dc3545")
    verdict = "✅ Network matches the design" if pct == 100 else ("⚠️ Network is partly working" if pct >= 70 else "🛑 Network does not match the design")
    return f'''<div style="{FONT}border:1px solid #ddd;border-radius:10px;padding:14px 18px;margin:8px 0;max-width:900px">
<div style="font-size:20px;font-weight:600">{verdict}</div>
<div style="color:#555;margin:2px 0 10px">{label} · {datetime.now():%d %b %Y %H:%M} · {elapsed:.1f}s</div>
<div style="background:#eee;border-radius:8px;height:18px;overflow:hidden"><div style="width:{pct}%;background:{color};height:18px"></div></div>
<div style="margin-top:8px;font-size:15px"><b style="color:#155724">{p} passed</b> &nbsp; <b style="color:#721c24">{f} failed</b> &nbsp; <b style="color:#856404">{e} unreachable</b> &nbsp; of {n} checks ({pct}%)</div></div>'''

def h_topology(df, inventory):
    W, H = 640, 230
    pairs, dead = bad_pairs(df, inventory), _dead_devices(df)
    def node_col(d):
        g = df[df.device == d] if not df.empty else df
        if len(g) == 0: return "#e2e3e5", "#6c757d"
        if (g.status == "ERROR").any(): return "#fff3cd", "#e0a800"
        if (g.status == "FAIL").any(): return "#f8d7da", "#dc3545"
        return "#d4edda", "#28a745"
    svg = [f'<svg viewBox="0 0 {W} {H}" width="{W}" height="{H}" style="{FONT}max-width:100%">']
    for a, b in LINKS:
        (x1, y1), (x2, y2) = inventory[a]["pos"], inventory[b]["pos"]
        if a in dead or b in dead: col, dash = "#e0a800", "6 5"
        elif frozenset([a, b]) in pairs: col, dash = "#dc3545", "6 5"
        elif inventory[a]["role"] == "switch": col, dash = "#6c757d", "2 4"
        else: col, dash = "#28a745", "0"
        svg.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{col}" stroke-width="4" stroke-dasharray="{dash}"/>')
    for name, spec in inventory.items():
        x, y = spec["pos"]; fill, stroke = node_col(name)
        g = df[df.device == name] if not df.empty else df
        sub = f"{(g.status == 'PASS').sum()}/{len(g)} ok" if len(g) else "–"
        svg.append(f'<rect x="{x-38}" y="{y-26}" width="76" height="52" rx="10" fill="{fill}" stroke="{stroke}" stroke-width="3"/>'
                   f'<text x="{x}" y="{y-3}" text-anchor="middle" font-size="16" font-weight="700">{name}</text>'
                   f'<text x="{x}" y="{y+16}" text-anchor="middle" font-size="12" fill="#444">{sub}</text>')
    svg.append(f'<text x="10" y="{H-8}" font-size="12" fill="#555">green line = OSPF healthy · red dashed = OSPF problem · amber dashed = device unreachable · dotted = switch uplink</text></svg>')
    return "<div>" + "".join(svg) + "</div>"

def h_diagnosis(items):
    if not items:
        return f'<div style="{FONT}padding:10px;color:#155724">Nothing to fix. 🎉</div>'
    out = [f'<div style="{FONT}max-width:900px"><div style="font-size:17px;font-weight:600;margin:10px 0 4px">What went wrong, and what to try</div>']
    for i in items:
        root = i["kind"] == "root"
        bar, bg, tag = ("#dc3545", "#fff5f5", "ROOT CAUSE") if root else ("#6c757d", "#f6f6f6", "SIDE-EFFECT")
        cmds = "".join(f'<div style="font-family:Consolas,monospace;background:#1e1e1e;color:#d4d4d4;padding:3px 8px;margin:2px 0;border-radius:4px;white-space:pre-wrap">{html.escape(c)}</div>' for c in i["fix"])
        out.append(f'<div style="border-left:5px solid {bar};background:{bg};padding:8px 12px;margin:8px 0;border-radius:6px">'
                   f'<span style="font-size:11px;color:{bar};font-weight:700">{tag}</span><br><b>{html.escape(i["title"])}</b>'
                   f'<div style="margin:4px 0;color:#333">{html.escape(i["why"])}</div>{cmds}</div>')
    return "".join(out) + "</div>"

def h_table(df):
    if df.empty: return ""
    rows = []
    for r in df.sort_values(["device", "category"]).itertuples(index=False):
        bg, fg, ic = STY[r.status]
        cells = "".join(f'<td style="padding:5px 9px;border:1px solid #ddd">{html.escape(str(v))}</td>' for v in (r.device, r.category, r.check, r.expected, r.actual))
        rows.append(f'<tr style="background:{bg};color:{fg}">{cells}<td style="padding:5px 9px;border:1px solid #ddd;font-weight:700">{ic} {r.status}</td></tr>')
    head = "".join(f'<th style="padding:6px 9px;background:#222;color:#fff;text-align:left">{c}</th>' for c in ["Device", "Check", "Item", "Expected", "Actual", "Result"])
    return f'<div style="{FONT}max-width:900px;margin-top:10px"><table style="border-collapse:collapse;width:100%;font-size:13px"><tr>{head}</tr>{"".join(rows)}</table></div>'

def render_all(df, elapsed, label, inventory, items):
    return h_scorecard(df, elapsed, label) + h_topology(df, inventory) + h_diagnosis(items) + h_table(df)

def save_report(df, elapsed, label, inventory, items):
    base = REPORT_DIR / f"build_check_{datetime.now():%Y%m%d_%H%M%S}"
    df.to_csv(f"{base}.csv", index=False)
    df.to_json(f"{base}.json", orient="records", indent=2)
    Path(f"{base}.html").write_text(f'<html><head><meta charset="utf-8"><title>Network build check</title></head><body style="margin:24px">'
                                    f'<h2 style="{FONT}">Network Build Check</h2>{render_all(df, elapsed, label, inventory, items)}</body></html>', encoding="utf-8")
    return base

### Safety net: automatic self-tests
These prove the parsers and the virtual network behave like a real one (a cable cut should kill the routes *and* the pings; a dead router should show as unreachable, not crash the run).

In [10]:
def _run(faults=()):
    net = VirtualNetwork()
    for f in faults: FAULTS[f][1](net)
    return run_checks(SimBackend(net), DESIGN_SIM)[0]

DESIGN_SIM = learn_design(SimBackend(VirtualNetwork()), INVENTORY)
(WORKDIR / "design_sim.json").write_text(json.dumps(DESIGN_SIM, indent=2))

# parsers on the lab's own output
_n = VirtualNetwork()
assert parse_vlans(_n.cli("SW1", "show vlan brief")) == {1: "default", 10: "SALES", 20: "HR", 30: "SERVERS", 1002: "fddi-default"}
assert parse_ospf_neighbors(_n.cli("R2", "show ip ospf neighbor")) == {"1.1.1.1": "FULL", "3.3.3.3": "FULL"}
assert parse_routes(_n.cli("R1", "show ip route"))["192.168.30.0/24"] == "O"
assert parse_ping(_n.cli("R1", "ping 192.168.30.1 repeat 5")) == 100

# behaviour of the lab
assert (_run().status == "PASS").all(), "healthy lab must pass everything"
d = _run(["cable"]);  assert (d[(d.device == "R1") & (d.category == "Route")].status == "FAIL").all()
d = _run(["mtu"]);    assert d[(d.device == "R2") & (d.check == "Neighbor 3.3.3.3")].actual.iloc[0] == "EXSTART"
d = _run(["hello"]);  assert d[(d.device == "R1") & (d.check == "to 10.0.12.2")].status.iloc[0] == "PASS"   # directly connected still pings
assert d[(d.device == "R1") & (d.check == "to 192.168.30.1")].status.iloc[0] == "FAIL"                        # but nothing beyond
d = _run(["r3off"]);  assert (d[d.device == "R3"].status == "ERROR").all()
print("✅ All self-tests passed. The design learned from the healthy lab has",
      sum(len(v.get("vlans", {})) + len(v.get("ospf_neighbors", [])) + len(v.get("routes", {})) + len(v.get("pings", [])) for v in DESIGN_SIM.values()), "checks.")

✅ All self-tests passed. The design learned from the healthy lab has 25 checks.


## 7. Quick tour (no clicking needed)
Three situations run automatically so you can see what the tool tells you. Read the *diagnosis* under each scorecard.

In [11]:
def tour(title, faults=()):
    net = VirtualNetwork()
    for f in faults: FAULTS[f][1](net)
    df, el = run_checks(SimBackend(net), DESIGN_SIM)
    display(HTML(f'<h3 style="{FONT}margin-bottom:0">{title}</h3>' + h_scorecard(df, el, "Practice lab") + h_topology(df, INVENTORY) + h_diagnosis(diagnose(df, INVENTORY))))

tour("Scenario A: everything built correctly")

In [12]:
tour("Scenario B: MTU mismatch on the R2–R3 link", ["mtu"])

In [13]:
tour("Scenario C: R3 powers off AND someone deleted VLAN 30", ["r3off", "vmiss"])

Notice in scenario B: **one** real problem (an MTU mismatch) produced twelve red rows, but the tool reports it as a single root cause plus a note that the rest are side-effects. That is the difference between a checker and a pile of `show` commands.

## 8. The dashboard (click instead of code)
**Practice lab:** choose a fault → 💥 Break it → ✅ Run checks → 🔧 Repair all.
**My real devices:** switch the toggle, paste/edit your inventory, type your login, press 🧠 Learn design on a *healthy* network once, then ✅ Run checks any time (before and after every change).

In [14]:
# GUI-CELL
import ipywidgets as w

NET = VirtualNetwork()
STATE = {"last": None, "active": []}

mode = w.ToggleButtons(options=[("🧪 Practice lab (no devices needed)", "sim"), ("🔌 My real devices", "live")], value="sim")
fault_dd = w.Dropdown(options=[(lbl, key) for key, (lbl, _) in FAULTS.items()], description="Fault:", layout=w.Layout(width="420px"))
b_break  = w.Button(description="💥 Break it", button_style="warning")
b_repair = w.Button(description="🔧 Repair all", button_style="info")
b_learn  = w.Button(description="🧠 Learn design", tooltip="Record what a healthy network looks like")
b_run    = w.Button(description="✅ Run checks", button_style="success", layout=w.Layout(width="160px"))
b_save   = w.Button(description="💾 Save report")
status   = w.HTML()
out      = w.Output()

live_user  = w.Text(description="Username", placeholder="or set NET_USER")
live_pass  = w.Password(description="Password", placeholder="or set NET_PASS")
live_en    = w.Password(description="Enable", placeholder="optional")
live_inv   = w.Textarea(value=json.dumps(INVENTORY, indent=1), layout=w.Layout(width="100%", height="230px"))
live_panel = w.VBox([w.HTML("<b>Your devices</b> (JSON: edit host / port / device_type, e.g. <code>cisco_ios_telnet</code> with the GNS3 console port):"),
                     live_inv, w.HBox([live_user, live_pass, live_en])], layout=w.Layout(display="none"))
sim_panel  = w.VBox([w.HTML("<b>Break something in the virtual network</b> (you can break several things at once):"), w.HBox([fault_dd, b_break, b_repair])])

def say(msg, color="#333"): status.value = f'<div style="{FONT}color:{color};padding:4px 0">{msg}</div>'

def backend_and_files():
    if mode.value == "sim":
        return SimBackend(NET), INVENTORY, WORKDIR / "design_sim.json"
    inv = json.loads(live_inv.value)
    return (LiveBackend(inv, live_user.value or os.environ.get("NET_USER", ""), live_pass.value or os.environ.get("NET_PASS", ""),
                        live_en.value or os.environ.get("NET_ENABLE", "")), inv, WORKDIR / "design_live.json")

def on_mode(_):
    sim_panel.layout.display = "" if mode.value == "sim" else "none"
    live_panel.layout.display = "" if mode.value == "live" else "none"
    out.clear_output(); say("Switched to " + ("the practice lab." if mode.value == "sim" else "real devices. Fill in the inventory and login, then press 🧠 Learn design on a healthy network."))
mode.observe(on_mode, "value")

def on_break(_):
    key = fault_dd.value
    FAULTS[key][1](NET)
    if key not in STATE["active"]: STATE["active"].append(key)
    say("💥 Active faults: " + ", ".join(FAULTS[k][0] for k in STATE["active"]) + "<br>Now press ✅ Run checks.", "#856404")

def on_repair(_):
    NET.reset(); STATE["active"].clear(); say("🔧 Everything repaired. Press ✅ Run checks to confirm.", "#155724")

def on_learn(_):
    with out:
        clear_output()
        try:
            be, inv, path = backend_and_files()
            src = SimBackend(VirtualNetwork()) if mode.value == "sim" else be     # lab: learn from a healthy copy
            design = learn_design(src, inv); path.write_text(json.dumps(design, indent=2))
            n = sum(len(v.get("vlans", {})) + len(v.get("ospf_neighbors", [])) + len(v.get("routes", {})) + len(v.get("pings", [])) for v in design.values())
            say(f"🧠 Learned a design with {n} checks → saved to {path.name}. You may open and edit it.", "#155724")
        except Exception as e:
            say(f"Could not learn the design: {html.escape(str(e))}", "#721c24")

def on_run(_):
    with out:
        clear_output()
        try:
            be, inv, path = backend_and_files()
            if not path.exists():
                if mode.value == "sim": path.write_text(json.dumps(DESIGN_SIM, indent=2))
                else:
                    say("Press 🧠 Learn design first (on a healthy network) so I know what to compare against.", "#856404"); return
            say("Running checks…")
            df, el = run_checks(be, json.loads(path.read_text()))
            items = diagnose(df, inv); STATE["last"] = (df, el, be.label, inv, items)
            say(""); display(HTML(render_all(df, el, be.label, inv, items)))
        except Exception as e:
            say(f"Something went wrong: {html.escape(str(e))}", "#721c24")

def on_save(_):
    if not STATE["last"]: say("Run the checks first.", "#856404"); return
    base = save_report(*STATE["last"]); say(f"💾 Saved {base.name}.html / .csv / .json in {REPORT_DIR}", "#155724")

b_break.on_click(on_break); b_repair.on_click(on_repair); b_learn.on_click(on_learn); b_run.on_click(on_run); b_save.on_click(on_save)
display(w.VBox([mode, sim_panel, live_panel, w.HBox([b_learn, b_run, b_save]), status, out]))

## 9. Using it on real equipment

1. Make sure each device allows SSH (or Telnet for GNS3/EVE-NG consoles) and has a user that can run `show` and `ping`.
2. In the dashboard choose **🔌 My real devices**, edit the JSON (`host`, `port`, `device_type`; for GNS3 use `cisco_ios_telnet`, host `127.0.0.1` and the console port).
3. Make sure the network is **healthy**, then press **🧠 Learn design**. Open `netcheck_data/design_live.json` and tidy it if you like.
4. Press **✅ Run checks** before and after every change. Save the report as evidence.

**Habits that make this a real project, not a lab**
- Run it *before* a change and *after*: any new red row is caused by your change.
- Keep each saved report as proof of the state of the network on that date.
- Keep `design_live.json` in Git so the design has a history.

## 10. Where to take it next
- **More checks:** interface up/up, STP root bridge, trunk-allowed VLANs, ACL counters, OSPF timers.
- **Speed:** log in to many devices in parallel (`concurrent.futures`) for large networks.
- **Automation:** schedule it (cron / Task Scheduler) and send a Telegram or email message when anything turns red.
- **Machine learning link:** every run produces labelled data: *(fault injected → pattern of failing checks)*. Use the practice lab to generate thousands of such examples and train a classifier (even a small neural network) to predict the fault from the check pattern; compare it with the rule-based `diagnose()` above.